# SynthBA Brain Age — IXI batch

**Before running:** Runtime → Change runtime type → **GPU (T4)**

Puglisi et al. 2024. Built-in preprocessing (SynthSeg + MNI). Pass raw T1.

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone / pull repo
import os
REPO = '/content/faceage-to-brainage'
if not os.path.exists(REPO):
    os.system(f'git clone https://github.com/kondratevakate/faceage-to-brainage.git {REPO}')
else:
    os.system(f'git -C {REPO} pull --rebase')
os.chdir(REPO)
print('Working dir:', os.getcwd())

In [ ]:
# 3. Install (no restart needed — only pure-Python / pre-built wheels)
import subprocess, sys
for pkg in ['synthba', 'nibabel', 'pandas', 'tqdm']:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True, text=True)
    print(f'  {pkg}:', 'ok' if r.returncode == 0 else r.stderr[-100:])
print('Done')

In [ ]:
# 4. Config — edit paths here
DRIVE        = '/content/drive/MyDrive'
IXI_MANIFEST = f'{DRIVE}/brain_mri/t1_only/ixi/manifest.csv'
OUT_CSV      = '/tmp/synthba_predictions.csv'   # saved to Drive at the end
DRIVE_OUT    = f'{DRIVE}/brain_mri/brainage_results/ixi/synthba_predictions.csv'
DEVICE       = 'cuda'   # 'cpu' if no GPU

import pandas as pd
from pathlib import Path
df = pd.read_csv(IXI_MANIFEST)
print(f'Manifest: {len(df)} scans')
print(df[['subject_id', 't1_path', 'age']].head(3).to_string())

In [ ]:
# 5. Run SynthBA — inline loop, no subprocess
import sys, gc, math
import torch
import nibabel as nib
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from synthba import SynthBA

OUT_CSV_PATH = Path('/tmp/synthba_predictions.csv')

# Resume: skip already done
done = set()
records = []
if OUT_CSV_PATH.exists():
    prev = pd.read_csv(OUT_CSV_PATH)
    done = set(prev[prev['status'] == 'ok']['subject_id'].astype(str))
    records = prev.to_dict('records')
    print(f'Resume: skipping {len(done)} already done')

manifest = pd.read_csv(IXI_MANIFEST)

for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc='SynthBA'):
    sid = str(row['subject_id'])
    if sid in done:
        continue

    record = {
        'subject_id': sid,
        'chron_age': float(row['age']) if not pd.isna(row['age']) else float('nan'),
        'predicted_age': float('nan'),
        'brain_age_gap': float('nan'),
        'model_name': 'SynthBA',
        'status': 'error',
        'error': '',
    }
    try:
        img = nib.load(str(row['t1_path']))
        sba = SynthBA(DEVICE)
        pred = float(sba.run(img, preprocess=True, mr_weighting='t1'))
        del sba; gc.collect()
        if DEVICE == 'cuda': torch.cuda.empty_cache()

        record['predicted_age'] = pred
        if not math.isnan(record['chron_age']):
            record['brain_age_gap'] = pred - record['chron_age']
        record['status'] = 'ok'
    except Exception as e:
        record['error'] = str(e)

    records.append(record)
    pd.DataFrame(records).to_csv(OUT_CSV_PATH, index=False)

result_df = pd.read_csv(OUT_CSV_PATH)
ok = (result_df['status'] == 'ok').sum()
print(f'Done: {ok}/{len(result_df)} ok')
print(result_df[['subject_id','chron_age','predicted_age','brain_age_gap','status']].head(5).to_string())

In [ ]:
# 6. Copy result to Drive
import shutil
from pathlib import Path
Path(DRIVE_OUT).parent.mkdir(parents=True, exist_ok=True)
shutil.copy('/tmp/synthba_predictions.csv', DRIVE_OUT)
print('Saved to Drive:', DRIVE_OUT)